# TaskNest Support Ticket Router

A LangGraph demonstration that classifies each ticket once, routes it to a support team, and returns a fixed acknowledgement. Add `OPENAI_API_KEY` to Google Colab Secrets before running.

In [ ]:
!pip -q install langgraph langchain-openai pydantic

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

MODEL_NAME = "gpt-6-luna"

class SupportState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "general"]
    team: str
    reply: str

class Classification(BaseModel):
    category: Literal["billing", "technical", "general"] = Field(
        description="The best matching TaskNest support category"
    )


## Classifier and graph nodes

The classifier performs the graph’s only LLM call. Team assignment and reply generation are fixed Python logic.

In [ ]:
classifier = ChatOpenAI(model=MODEL_NAME, temperature=0).with_structured_output(Classification)

CLASSIFICATION_INSTRUCTIONS = """Classify the TaskNest support ticket by the user's primary intent.
Route payments, invoices, refunds, and subscriptions to billing.
Route bugs, crashes, errors, and login issues to technical.
Route feature requests, general questions, and thank-you notes to general.
Prioritize intent over simple keyword matches. For example, an app crash occurring
while the user tries to pay is a technical issue because the primary reported problem
is the crash. Return only the structured category."""

def classify(state: SupportState) -> dict:
    result = classifier.invoke([
        ("system", CLASSIFICATION_INSTRUCTIONS),
        ("human", state["ticket"]),
    ])
    return {"category": result.category}

def route(state: SupportState) -> str:
    return state["category"]

def billing_team(state: SupportState) -> dict:
    return {
        "team": "Billing Team",
        "reply": "Thanks for contacting us. Our Billing Team will review your request.",
    }

def tech_team(state: SupportState) -> dict:
    return {
        "team": "Tech Support Team",
        "reply": "Thanks for letting us know. Our Tech Support Team will look into this.",
    }

def general_team(state: SupportState) -> dict:
    return {
        "team": "General Support Team",
        "reply": "Thanks for reaching out. Our General Support Team is here to help.",
    }


In [ ]:
builder = StateGraph(SupportState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route,
    {
        "billing": "billing_team",
        "technical": "tech_team",
        "general": "general_team",
    },
)
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)

app = builder.compile()


## Graph visualization

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))


## Example tickets

In [ ]:
test_tickets = [
    "I was charged twice for my monthly subscription.",
    "Could you send me a copy of last month's invoice?",
    "I get an error when I try to log in.",
    "The mobile app freezes whenever I open the dashboard.",
    "Could you add a dark mode feature?",
    "Thank you for helping me set up my workspace!",
    "The app crashes every time I try to pay",
]

for ticket in test_tickets:
    result = app.invoke({
        "ticket": ticket,
        "category": "general",
        "team": "",
        "reply": "",
    })
    print(f"ticket: {result['ticket']}")
    print(f"category: {result['category']}")
    print(f"team: {result['team']}")
    print(f"reply: {result['reply']}")
    print("-")
